In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing import image

# For training improvements
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint  # Control training
from tensorflow.keras.layers import BatchNormalization                # Normalize activations

In [2]:
# Path to dataset directory (should contain subfolders like 'with_mask', 'without_mask')
train_dir = 'data/'

In [3]:
# ImageDataGenerator helps in Utility to load, preprocess, Normalizing images, Augmenting data (rotation, zoom, flip)
train_gen = ImageDataGenerator(
    rescale = 1./255,           
    validation_split = 0.2,     
    rotation_range = 20,        
    zoom_range = 0.2,           
    horizontal_flip = True      
)


# Go to my dataset folder, load images in batches, apply preprocessing & augmentation, and assign labels automatically
train_data = train_gen.flow_from_directory(
    train_dir,
    # Model cannot handle different sizes so we can Resize all images to 128x128 fixed size which can give more detail and speed
    target_size = (128, 128),   
    batch_size = 32,            
    class_mode = 'binary',      
    subset = 'training'         
)


# Load validation data
val_data = train_gen.flow_from_directory(
    train_dir,
    target_size=(128, 128),  
    batch_size=32,
    class_mode='binary',
    subset='validation'       
)

Found 6043 images belonging to 3 classes.
Found 1510 images belonging to 3 classes.


In [4]:
# Display class label mapping (e.g., {'with_mask': 0, 'without_mask': 1})
print("Class Labels:", train_data.class_indices)

Class Labels: {'.ipynb_checkpoints': 0, 'with_mask': 1, 'without_mask': 2}


In [5]:
model = Sequential([

    # First Convolution Block
    
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),  # Extract low-level features
    BatchNormalization(),   # Normalize outputs → stabilizes training

    # Reduce spatial size (downsampling)
    # reduce size + keep important features
    MaxPooling2D(2,2),      

    # Second Convolution Block
    Conv2D(64, (3,3), activation='relu'),  # Learn more complex features
    BatchNormalization(),   # Normalize again
    MaxPooling2D(2,2),      # Downsample

    # Third Convolution Block
    Conv2D(128, (3,3), activation='relu'), # High-level feature extraction
    BatchNormalization(),   # Normalize
    MaxPooling2D(2,2),      # Downsample

    Flatten(),

    # Fully connected layer
    Dense(128, activation='relu'),  # Learn patterns from extracted features
    
    Dropout(0.5),                   

    # Output layer (binary classification)
    Dense(1, activation='sigmoid')  # Output probability (0 to 1) it will give 0    or 1 (mask or no mask)
])

In [6]:
# Compile model (configure training)
model.compile(
    optimizer = Adam( learning_rate=0.001 ),  # Optimizer with learning rate
    loss='binary_crossentropy',               # Loss for binary classification
    metrics=['accuracy']                      # Track accuracy
)

In [7]:
# EarlyStopping stops training if validation loss doesn't improve
early_stop = EarlyStopping(
    monitor = 'val_loss',           # Watch validation loss
    patience = 3,                   # Wait 3 epochs before stopping
    restore_best_weights = True     # Restore best model weights
)

# ModelCheckpoint saves the best model during training
checkpoint = ModelCheckpoint(
    "best_model.keras",             # File name
    monitor = 'val_accuracy',       # Watch validation accuracy
    save_best_only = True           # Save only best version
)

In [9]:
# Train the model
history = model.fit(
    train_data,                          # Training data generator
    epochs = 10,                           # Number of training cycles
    validation_data = val_data,            # Validation data

    
    steps_per_epoch = len(train_data),     # Number of batches per epoch
    validation_steps = len(val_data),      # Validation batches

    callbacks = [early_stop, checkpoint]   # Callbacks are tools that monitor training and take action automatically.
)

Epoch 1/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 301s 1s/step - accuracy: 0.4784 - loss: -43137.2344 - val_accuracy: 0.4934 - val_loss: -68529.7500
Epoch 2/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 281s 1s/step - accuracy: 0.4928 - loss: -486156.2500 - val_accuracy: 0.4934 - val_loss: -566055.8125
Epoch 3/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 278s 1s/step - accuracy: 0.4930 - loss: -1931223.1250 - val_accuracy: 0.4934 - val_loss: -2320010.7500
Epoch 4/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.4931 - loss: -4926122.0000 - val_accuracy: 0.4934 - val_loss: -7958246.0000
Epoch 5/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 252s 1s/step - accuracy: 0.4931 - loss: -9927826.0000 - val_accuracy: 0.4934 - val_loss: -10322939.0000
Epoch 6/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 270s 1s/step - accuracy: 0.4931 - loss: -17335962.0000 - val_accuracy: 0.4934 - val_loss: -10834557.0000
Epoch 7/10
189/189 ━━━━━━━━━━━━━━━━━━━━ 333s 1s/step - accuracy: 0.4931 - loss: -27554618.0000 - val_accuracy: 0.4934 - val_loss: -26971612.0000


In [ ]:




# Plot training results (accuracy & loss)

plt.figure(figsize=(10,4))

# Plot accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')       # Training accuracy
plt.plot(history.history['val_accuracy'], label='Val Accuracy')     # Validation accuracy
plt.title("Accuracy")
plt.legend()

# Plot loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')               # Training loss
plt.plot(history.history['val_loss'], label='Val Loss')             # Validation loss
plt.title("Loss")
plt.legend()

plt.show()

In [10]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "with_mask_1.jpg"

img = image.load_img(img_path, target_size=(128,128))
img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print("Without Mask 😷❌")
else:
    print("With Mask 😷✅")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
Without Mask 😷❌


In [ ]:
# Save trained model to file
model.save("mask_final.keras")

print("Model saved successfully!")